# Dataset Kendaraan: 5 Kelas Terpisah

Mengambil **motor, sepeda, mobil, truck, bus** dari COCO sebagai kelas terpisah (bukan digabung).
Batas dikontrol per **gambar** (bukan per instance) agar distribusi dataset lebih seimbang.

| ID | Kelas | Nama di COCO |
|---|---|---|
| 0 | car | car |
| 1 | motorcycle | motorcycle |
| 2 | bicycle | bicycle |
| 3 | truck | truck |
| 4 | bus | bus |


In [ ]:
# ============================================================
# KONFIGURASI — ubah sesuai kebutuhan
# ============================================================
PER_CLASS_MAX_IMAGES = {
    'car':        1500,   # paling banyak di COCO, boleh lebih
    'motorcycle': 1000,
    'bicycle':    800,
    'truck':      800,
    'bus':        600,
}
VEHICLE_MIN_PX = 50      # buang kotak < 50px (terlalu jauh/kabur)

# Urutan final kelas → menentukan ID di data.yaml
FINAL_CLASSES = ['car', 'motorcycle', 'bicycle', 'truck', 'bus']

OUTPUT_DIR   = '/content/vehicle_dataset'
DRIVE_DIR    = '/content/drive/MyDrive/YOLO/vehicle_dataset'


In [ ]:
!pip install -q python-dotenv roboflow pyyaml
from google.colab import drive
drive.mount('/content/drive')

import os
from dotenv import load_dotenv
load_dotenv('/content/drive/MyDrive/YOLO/vnetra.env')

import shutil, glob, random, yaml
from roboflow import Roboflow
import PIL; import importlib, site
importlib.reload(site)

os.makedirs(DRIVE_DIR, exist_ok=True)
print('Setup selesai!')


In [ ]:
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])

print('Mengunduh COCO...')
dataset_coco = rf.workspace('microsoft').project('coco').version(46).download('yolov11')
print(f'COCO siap: {dataset_coco.location}')


In [ ]:
# ============================================================
# Bangun mapping: nama kelas COCO → ID COCO
# ============================================================
coco_yaml_path = os.path.join(dataset_coco.location, 'data.yaml')
with open(coco_yaml_path) as f:
    coco_meta = yaml.safe_load(f)

coco_names = coco_meta.get('names', [])
if isinstance(coco_names, dict):
    coco_names = [coco_names[i] for i in range(len(coco_names))]

# { 'car': 2, 'motorcycle': 3, ... } — ID dari COCO asli
COCO_ID = {name.lower(): i for i, name in enumerate(coco_names)}

# Hanya kelas yang kita mau
target_coco_ids = {}
for cls in FINAL_CLASSES:
    if cls in COCO_ID:
        target_coco_ids[COCO_ID[cls]] = cls
    else:
        print(f'PERINGATAN: "{cls}" tidak ada di COCO!')

# Mapping dari nama kelas → ID final kita
FINAL_ID = {cls: i for i, cls in enumerate(FINAL_CLASSES)}

print('Target kelas dari COCO:')
for coco_id, cls in target_coco_ids.items():
    print(f'  COCO ID {coco_id} ({cls}) → Final ID {FINAL_ID[cls]}')

# ============================================================
# Buat folder output
# ============================================================
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
for split in ['train', 'valid', 'test']:
    os.makedirs(f'{OUTPUT_DIR}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUTPUT_DIR}/{split}/labels', exist_ok=True)

# ============================================================
# Proses setiap split
# ============================================================
img_count = {cls: 0 for cls in FINAL_CLASSES}  # berapa gambar per kelas
total_written = {'train': 0, 'valid': 0, 'test': 0}

for split in ['train', 'valid', 'test']:
    src_img = os.path.join(dataset_coco.location, split, 'images')
    src_lbl = os.path.join(dataset_coco.location, split, 'labels')
    if not os.path.exists(src_img):
        continue

    all_images = glob.glob(f'{src_img}/*')
    random.seed(42)
    random.shuffle(all_images)

    for img_path in all_images:
        fname = os.path.basename(img_path)
        lbl_name = fname.rsplit('.', 1)[0] + '.txt'
        lbl_path = os.path.join(src_lbl, lbl_name)
        if not os.path.exists(lbl_path):
            continue

        # Baca label, filter hanya kelas target + ukuran minimum
        new_lines = []
        classes_in_image = set()

        for line in open(lbl_path):
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            coco_id = int(parts[0])
            if coco_id not in target_coco_ids:
                continue  # bukan kendaraan target, buang
            # Filter ukuran minimum
            w, h = float(parts[3]), float(parts[4])
            if max(w, h) * 640 < VEHICLE_MIN_PX:
                continue
            cls_name = target_coco_ids[coco_id]
            final_id = FINAL_ID[cls_name]
            new_lines.append(f"{final_id} {' '.join(parts[1:])}\n")
            classes_in_image.add(cls_name)

        if not new_lines:
            continue

        # Cek apakah ada kelas yang masih perlu gambar
        # (hanya berlaku di split 'train'; valid/test tidak dibatasi)
        if split == 'train':
            if not any(
                img_count[cls] < PER_CLASS_MAX_IMAGES[cls]
                for cls in classes_in_image
            ):
                continue  # semua kelas dalam gambar ini sudah penuh

        # Tulis gambar + label
        shutil.copy(img_path, f'{OUTPUT_DIR}/{split}/images/{fname}')
        with open(f'{OUTPUT_DIR}/{split}/labels/{lbl_name}', 'w') as f:
            f.writelines(new_lines)

        # Update counter (hanya untuk train)
        if split == 'train':
            for cls in classes_in_image:
                img_count[cls] += 1

        total_written[split] += 1

print('Selesai memproses COCO!')


In [ ]:
# Tulis data.yaml
with open(f'{OUTPUT_DIR}/data.yaml', 'w') as f:
    yaml.dump({
        'path': OUTPUT_DIR,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': len(FINAL_CLASSES),
        'names': FINAL_CLASSES,
    }, f, sort_keys=False)

# Hitung instance per kelas
def count_instances(label_dir, nc):
    counts = {i: 0 for i in range(nc)}
    if not os.path.exists(label_dir): return counts
    for lf in os.listdir(label_dir):
        if not lf.endswith('.txt'): continue
        for line in open(os.path.join(label_dir, lf)):
            p = line.strip().split()
            if p: counts[int(p[0])] += 1
    return counts

nc = len(FINAL_CLASSES)
tr = count_instances(f'{OUTPUT_DIR}/train/labels', nc)
va = count_instances(f'{OUTPUT_DIR}/valid/labels', nc)
te = count_instances(f'{OUTPUT_DIR}/test/labels', nc)

def n_img(d): return len(glob.glob(f'{d}/*')) if os.path.exists(d) else 0

print('=' * 60)
print(f'Total gambar  | Train: {total_written["train"]:,} | Valid: {total_written["valid"]:,} | Test: {total_written["test"]:,}')
print()
print(f'{"ID":<4} {"Kelas":<12} {"Train img":>10} {"Train inst":>11} {"Val inst":>9} {"Test inst":>9}')
print('-' * 60)
for i, cls in enumerate(FINAL_CLASSES):
    print(f'{i:<4} {cls:<12} {img_count[cls]:>10,} {tr[i]:>11,} {va[i]:>9,} {te[i]:>9,}')
print('=' * 60)


In [ ]:
zip_path = f'{DRIVE_DIR}/vehicle_dataset'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
size_mb = os.path.getsize(f'{zip_path}.zip') / 1024**2
print(f'Tersimpan: {zip_path}.zip ({size_mb:.1f} MB)')
print('Upload file ini ke Kaggle bersama vnetra_master_dataset.zip untuk training.')
